# Crisis Prediction Model Training Pipeline

In [1]:
!pip install --upgrade pip
!pip install -r requirements.txt

In [3]:
import sys

# Add the scripts directory to sys.path
sys.path.append('/home/sagemaker-user/ACAPS/src/task-3-modelling/scripts')

# Verify that the path is added
print(sys.path)


['/root/ACAPS/src/task-3-modelling/scripts', '/usr/local/lib/python310.zip', '/usr/local/lib/python3.10', '/usr/local/lib/python3.10/lib-dynload', '', '/usr/local/lib/python3.10/site-packages', '/home/sagemaker-user/ACAPS/src/task-3-modelling/scripts']


In [9]:
# Import required libraries
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, ParameterGrid

# Import custom modules
from scripts.config import Config
from scripts.data_loader import DataLoader
from scripts.preprocessing import DataPreprocessor
from scripts.feature_engineering import FeatureEngineering
from scripts.feature_importance import FeatureSelector
from scripts.model_architecture import CrisisPredictor
from scripts.training import ModelTrainer
from scripts.evaluation import ModelEvaluator
from scripts.uncertainty import UncertaintyEstimator
from scripts.visualization import Visualizer
from scripts.model_persistence import ModelPersistence

ModuleNotFoundError: No module named 'models'

# 1. Initialize Configuration

In [ ]:
config = Config('config.yaml')
print("Configuration loaded successfully")

# 2. Load and Prepare Data

In [ ]:
data_loader = DataLoader(config)
df = data_loader.load_data()
train_df, test_df = data_loader.split_temporal(df)
print(f"Training set size: {len(train_df)}, Test set size: {len(test_df)}")

# 2.1 Data Visualization and Analysis **(Experimental)**

In [ ]:
 def visualize_batch_data(train_df, test_df, config):
    plt.figure(figsize=(15, 10))

    # Plot distribution of target variables
    for target, specs in config.targets.items():
        plt.subplot(2, 2, 1)
        sns.histplot(data=train_df[specs['columns']], bins=30)
        plt.title(f'{target} Distribution in Training Data')

        # Time series plot
        plt.subplot(2, 2, 2)
        plt.plot(train_df['date'], train_df[specs['columns']], label='Train')
        plt.plot(test_df['date'], test_df[specs['columns']], label='Test')
        plt.title(f'{target} Time Series Split')
        plt.legend()

    # Feature correlations
    plt.subplot(2, 2, 3)
    feature_corr = train_df[list(config.feature_groups.values())[0]].corr()
    sns.heatmap(feature_corr, cmap='coolwarm')
    plt.title('Feature Correlations')

    # Data split proportions
    plt.subplot(2, 2, 4)
    splits = [len(train_df), len(test_df)]
    plt.pie(splits, labels=['Train', 'Test'], autopct='%1.1f%%')
    plt.title('Data Split Proportions')

    plt.tight_layout()
    plt.show()

visualize_batch_data(train_df, test_df, config)
print(f"Total samples: {len(df)}")
print(f"Training samples: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Testing samples: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# 3. Feature Engineering and Selection

In [ ]:
# Feature Engineering
feature_engineer = FeatureEngineering(config)
train_engineered = feature_engineer.process_features(train_df)
test_engineered = feature_engineer.process_features(test_df)

# Preprocessing
preprocessor = DataPreprocessor(config)
train_processed = preprocessor.process_data(train_engineered)
test_processed = preprocessor.process_data(test_engineered)

# Feature Selection
feature_selector = FeatureSelector(config)
X_train, y_train = data_loader.prepare_features_targets(train_processed)
selected_features = feature_selector.select_features(X_train, y_train)

# 3.5 Hyperparameter Tuning ** (Experimental)**

In [ ]:
param_grids = {
    'lightgbm': {
        'n_estimators': [500, 1000, 1500],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'num_leaves': [31, 63, 127]
    },
    'catboost': {
        'iterations': [500, 1000, 1500],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth': [4, 6, 8]
    },
    'lstm': {
        'hidden_size': [32, 50, 64],
        'num_layers': [1, 2],
        'dropout': [0.1, 0.2, 0.3]
    }
}

best_params = {}
for target in config.targets:
    best_params[target] = {}
    for model_type, param_grid in param_grids.items():
        print(f"Tuning {model_type} for {target}...")

        if model_type in ['lightgbm', 'catboost']:
            model_class = model.models[target][model_type].__class__
            grid_search = GridSearchCV(
                estimator=model_class(),
                param_grid=param_grid,
                cv=5,
                scoring='neg_mean_squared_error',
                n_jobs=-1
            )
            grid_search.fit(X_train[selected_features[target]], y_train[target])
            best_params[target][model_type] = grid_search.best_params_

        elif model_type == 'lstm':
            best_loss = float('inf')
            best_lstm_params = {}

            for params in ParameterGrid(param_grid):
                model.models[target]['lstm'].set_params(**params)
                val_loss = trainer.validate_lstm(X_train, y_train, target)

                if val_loss < best_loss:
                    best_loss = val_loss
                    best_lstm_params = params

            best_params[target]['lstm'] = best_lstm_params

model.update_parameters(best_params)
print("\nBest parameters found:")
for target, params in best_params.items():
    print(f"\n{target}:")
    for model_type, model_params in params.items():
        print(f"{model_type}: {model_params}")

# 4. Model Training

In [ ]:
# Initialize model
model = CrisisPredictor(config)
trainer = ModelTrainer(config, model)

# Train model
history = trainer.train_model(X_train, y_train)

# Visualize training history
visualizer = Visualizer(config)
visualizer.plot_training_history(history)

# 5. Model Evaluation and Uncertainty Estimation

In [ ]:
# Prepare test data
X_test, y_test = data_loader.prepare_features_targets(test_processed)

# Evaluate model
evaluator = ModelEvaluator(config)
metrics = evaluator.evaluate_predictions(y_test, model.forward(X_test))

# Estimate uncertainty
uncertainty_estimator = UncertaintyEstimator(config)
predictions, uncertainties = uncertainty_estimator.monte_carlo_dropout(model, X_test)

# Visualize results
visualizer.plot_results(test_df['date'], predictions, metrics)

# 6. Save Model and Results

In [ ]:
persistence = ModelPersistence(config)
metadata = {
    'training_history': history,
    'feature_importance': feature_selector.feature_importance,
    'performance_metrics': metrics
}
persistence.save_model(model, metadata, 'v1.0')
print("Model and results saved successfully")